In [ ]:
# ============================================================================
# PREREQUISITE SETUP - RUN THIS FIRST
# ============================================================================

# Install libraries
!pip install xgboost lightgbm catboost shap scikit-learn matplotlib seaborn plotly -q

# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve)
from sklearn.preprocessing import label_binarize
import joblib
import pickle
import json
from datetime import datetime
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

# Load your dataset (update the path)
df = pd.read_csv('/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Data set/processed data set new.csv')  # UPDATE THIS PATH

# ============================================================================
# MODIFICATION START: Sample 1000 rows
# ============================================================================
TARGET_ROWS = 1000
if len(df) > TARGET_ROWS:

    try:
        df_sampled = df.groupby('job_title', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(TARGET_ROWS * len(x) / len(df))), random_state=42)
        )
        # Ensure we don't exceed TARGET_ROWS due to rounding
        df_sampled = df_sampled.head(TARGET_ROWS)

        # If stratified sampling didn't get exactly 1000 rows or failed, fall back to simple sample
        if len(df_sampled) != TARGET_ROWS:
            df_sampled = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
            print(f" Fell back to simple random sampling for 1000 rows due to class imbalance.")

        df = df_sampled
    except Exception:
        df = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
        print(f"⚠️ Simple random sampling used for 1000 rows.")

print("\n" + "="*70)
print(f"DATASET OVERVIEW (REDUCED TO {len(df)} ROWS)")
print("="*70)
# ============================================================================
# MODIFICATION END
# ============================================================================

# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Define features and target
feature_cols = [
    'search_country_encoded', 'job_level_encoded', 'job_type_onsite',
    'job_title_word_count_scaled', 'skills_word_count_scaled',
    'job_title_char_count_scaled', 'skills_char_count_scaled',
    'skills_list_count_scaled', 'title_tokens_count_scaled',
    'skills_tokens_count_scaled', 'job_level_encoded_scaled',
    'search_country_encoded_scaled',
    'has_skill_communication', 'has_skill_customer_service',
    'has_skill_teamwork', 'has_skill_leadership',
    'has_skill_communication_skills'
] + [f'tfidf_component_{i}' for i in range(20)]

# Prepare X and y
X = df[feature_cols].fillna(0)
y = df['job_title']

# 2. Handle rare classes
class_counts = y.value_counts()

y = y.apply(lambda x: x if class_counts[x] >= 2 else "Other")

# 3. Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Features: {len(feature_cols)}")
print(f" Samples: {len(X)}") # Should be 1000
print(f"\n Target Distribution (after rare class handling):")
print(y.value_counts())


print(f"\n Training samples: {len(X_train)}") # Should be 800 (80% of 1000)
print(f" Test samples: {len(X_test)}")      # Should be 200 (20% of 1000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.0 MB/s eta 0:00:00
Mounted at /content/drive
 Fell back to simple random sampling for 1000 rows due to class imbalance.

DATASET OVERVIEW (REDUCED TO 1000 ROWS)
Shape: (1000, 66)

Columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_title_clean', 'company_clean', 'job_skills_clean', 'job_location_clean', 'job_title_word_count', 'skills_word_count', 'job_title_char_count', 'skills_char_count', 'skills_list_count', 'has_skill_communication', 'has_skill_customer_service', 'has_skill_teamwork', 'has_skill_leadership', 'has_skill_communication_skills', 'job_title_tokens', 'job_skills_tokens', 'title_tokens_count', 'skills_tokens_count', 'combined_text_clean', 'job_level_encoded', 'job_type_onsite', 'search_country_encoded', 'job_title_word_count_sc

In [ ]:
# ============================================================================
# MODEL 3: RANDOM FOREST CLASSIFIER
# ============================================================================

print("TRAINING RANDOM FOREST CLASSIFIER....")


from sklearn.ensemble import RandomForestClassifier

# Create model directory
model_dir = f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Notebooks/Model_03_Random_Forest_Classifier.ipynb'

TRAINING RANDOM FOREST CLASSIFIER....


In [ ]:
# ============================================================================
# PART 1: EDA BEFORE TRAINING
# ============================================================================

print("\nPART 1: PRE-TRAINING EDA")

# 1. Pairplot for selected features
selected_features = ['job_title_word_count_scaled', 'skills_word_count_scaled',
                     'skills_list_count_scaled', 'tfidf_component_0', 'tfidf_component_1']
pairplot_data = X_train[selected_features].copy()
pairplot_data['target'] = y_train.values

# Limit to 1000 samples for speed
if len(pairplot_data) > 1000:
    pairplot_data = pairplot_data.sample(n=1000, random_state=42)

# Save pairplot info (plotting takes time, so we'll create a simplified version)
plt.figure(figsize=(15, 12))
for i, feat1 in enumerate(selected_features):
    for j, feat2 in enumerate(selected_features):
        plt.subplot(len(selected_features), len(selected_features), i*len(selected_features) + j + 1)
        if i == j:
            plt.hist(pairplot_data[feat1], bins=30, alpha=0.7, color='purple', edgecolor='black')
            plt.ylabel('Frequency')
        else:
            plt.scatter(pairplot_data[feat2], pairplot_data[feat1], alpha=0.3, s=10)
        if i == len(selected_features) - 1:
            plt.xlabel(feat2, fontsize=8)
        if j == 0:
            plt.ylabel(feat1, fontsize=8)
plt.suptitle('Feature Relationships (Random Sample)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-feature_pairplot.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Correlation with Target (for encoded features)
numeric_features = ['job_title_word_count_scaled', 'skills_word_count_scaled',
                    'job_title_char_count_scaled', 'skills_char_count_scaled',
                    'skills_list_count_scaled', 'has_skill_communication',
                    'has_skill_customer_service', 'has_skill_teamwork']

correlation_with_target = []
for feat in numeric_features:
    corr = X_train[feat].corr(y_train.map({val: i for i, val in enumerate(y_train.unique())}))
    correlation_with_target.append({'feature': feat, 'correlation': corr})

corr_df = pd.DataFrame(correlation_with_target).sort_values('correlation', key=abs, ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(corr_df['feature'], corr_df['correlation'], color='indigo', edgecolor='black')
plt.xlabel('Correlation with Target', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Feature Correlation with Target', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-target_correlation.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Missing Values Analysis
missing_data = X_train.isnull().sum()
if missing_data.sum() > 0:
    missing_df = missing_data[missing_data > 0].sort_values(ascending=False)
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(missing_df)), missing_df.values, color='purple', edgecolor='black')
    plt.xticks(range(len(missing_df)), missing_df.index, rotation=45, ha='right')
    plt.xlabel('Features', fontsize=12, fontweight='bold')
    plt.ylabel('Missing Count', fontsize=12, fontweight='bold')
    plt.title('Missing Values per Feature', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-missing_values.png', dpi=300, bbox_inches='tight')
    plt.close()
else:
    print("✅ No missing values detected")

print("✅ Pre-training EDA visualizations saved!")


PART 1: PRE-TRAINING EDA
✅ No missing values detected
✅ Pre-training EDA visualizations saved!


In [ ]:
# ============================================================================
# PART 2: HYPERPARAMETER TUNING (3 ITERATIONS)
# ============================================================================

print("\nPART 2: HYPERPARAMETER TUNING (3 Iterations)")

param_grids = [
    # Iteration 1: Broad search
    {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt', 'log2']
    },
    # Iteration 2: Refined search
    {
        'n_estimators': [150, 200, 250],
        'max_depth': [15, 20, 25],
        'min_samples_split': [3, 5, 7],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt']
    },
    # Iteration 3: Fine-tuning
    {
        'n_estimators': [200, 250, 300],
        'max_depth': [18, 20, 22],
        'min_samples_split': [4, 5, 6],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt'],
        'bootstrap': [True, False]
    }
]

tuning_results = []

for iteration, param_grid in enumerate(param_grids, 1):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration}/3")
    print(f"{'='*60}")

    base_model = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=2
    )

    start_time = time.time()
    grid_search.fit(X_train, y_train)
    end_time = time.time()

    tuning_results.append({
        'iteration': iteration,
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'time_taken': end_time - start_time
    })

    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")
    print(f"Time Taken: {end_time - start_time:.2f}s")

with open(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-tuning_results.json', 'w') as f:
    json.dump(tuning_results, f, indent=4)

best_iteration = max(tuning_results, key=lambda x: x['best_score'])
best_params = best_iteration['best_params']

print(f"\nBest Overall Parameters (Iteration {best_iteration['iteration']}):")
print(json.dumps(best_params, indent=4))


PART 2: HYPERPARAMETER TUNING (3 Iterations)

Iteration 1/3
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV Score: 0.8246
Time Taken: 184.57s

Iteration 2/3
Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 200}
Best CV Score: 0.8238
Time Taken: 286.79s

Iteration 3/3
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters: {'bootstrap': False, 'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 250}
Best CV Score: 0.8331
Time Taken: 805.80s

Best Overall Parameters (Iteration 3):
{
    "bootstrap": false,
    "max_depth": 18,
    "max_features": "sqrt",
    "min_samples_leaf": 1,
    "min_samples_split": 4,
    "n_estimators": 25

In [ ]:
# ============================================================================
# PART 3: CROSS-VALIDATION WITH BEST PARAMETERS
# ============================================================================

print("\nPART 3: CROSS-VALIDATION (5-Fold)")

final_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(
    final_model, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1
)

print(f"\nCV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), cv_scores, marker='D', linestyle='-', linewidth=2, markersize=10, color='purple')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.fill_between(range(1, 6), cv_scores.mean() - cv_scores.std(),
                 cv_scores.mean() + cv_scores.std(), alpha=0.2, color='purple')
plt.xlabel('Fold', fontsize=12, fontweight='bold')
plt.ylabel('F1 Score', fontsize=12, fontweight='bold')
plt.title('Random Forest Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-cv_scores.png', dpi=300, bbox_inches='tight')
plt.close()



PART 3: CROSS-VALIDATION (5-Fold)


NameError: name 'best_params' is not defined

In [ ]:
# ============================================================================
# PART 4: FINAL MODEL TRAINING
# ============================================================================

print("\nPART 4: FINAL MODEL TRAINING")

start_time = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"Model trained in {train_time:.2f}s")

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)

train_metrics = {
    'accuracy': accuracy_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
}

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
}

print("\nTRAINING SET METRICS:")
for metric, value in train_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

print("\nTEST SET METRICS:")
for metric, value in test_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")


PART 4: FINAL MODEL TRAINING
Model trained in 2.04s

TRAINING SET METRICS:
  Accuracy: 1.0000
  Precision: 1.0000
  Recall: 1.0000
  F1_score: 1.0000

TEST SET METRICS:
  Accuracy: 0.8800
  Precision: 0.7920
  Recall: 0.8800
  F1_score: 0.8325


In [ ]:
# ============================================================================
# PART 5: POST-TRAINING VISUALIZATIONS
# ============================================================================

print("\n📊 PART 5: POST-TRAINING VISUALIZATIONS")

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=True)
plt.title('Random Forest Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='mediumpurple', edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Random Forest Top 20 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

feature_importance.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-feature_importance.csv', index=False)

# 3. Tree Depth Analysis
tree_depths = [estimator.tree_.max_depth for estimator in final_model.estimators_]
plt.figure(figsize=(12, 6))
plt.hist(tree_depths, bins=30, color='purple', alpha=0.7, edgecolor='black')
plt.xlabel('Tree Depth', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title(f'Distribution of Tree Depths (Mean: {np.mean(tree_depths):.1f})', fontsize=14, fontweight='bold')
plt.axvline(x=np.mean(tree_depths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(tree_depths):.1f}')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-tree_depths.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. Metrics Comparison
metrics_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Train': [train_metrics['accuracy'], train_metrics['precision'],
              train_metrics['recall'], train_metrics['f1_score']],
    'Test': [test_metrics['accuracy'], test_metrics['precision'],
             test_metrics['recall'], test_metrics['f1_score']]
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, metrics_comparison['Train'], width, label='Train', color='plum', edgecolor='black')
bars2 = ax.bar(x + width/2, metrics_comparison['Test'], width, label='Test', color='purple', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Random Forest: Train vs Test Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_comparison['Metric'])
ax.legend()
ax.set_ylim([0, 1.1])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 03 - outputs-metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("✅ All visualizations saved!")


📊 PART 5: POST-TRAINING VISUALIZATIONS
✅ All visualizations saved!


In [ ]:
# ============================================================================
# PART 6: SAVE MODEL AND REPORTS
# ============================================================================

print("\nPART 6: SAVING MODEL AND REPORTS")

joblib.dump(final_model, f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-randomforest_model.pkl')
print(f"Model saved: /content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-randomforest_model.pkl")

report = classification_report(y_test, y_test_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-classification_report.csv')
print(f"Classification report saved")

results_summary = {
    'model_name': 'Random Forest Classifier',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'hyperparameter_tuning': tuning_results,
    'best_parameters': best_params,
    'cross_validation': {
        'cv_scores': cv_scores.tolist(),
        'mean_cv_score': float(cv_scores.mean()),
        'std_cv_score': float(cv_scores.std())
    },
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'training_time': train_time,
    'feature_importance_top10': feature_importance.head(10).to_dict('records'),
    'tree_statistics': {
        'mean_depth': float(np.mean(tree_depths)),
        'max_depth': int(np.max(tree_depths)),
        'min_depth': int(np.min(tree_depths))
    }
}

with open(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=4)

print(f"Results summary saved")

predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_test_pred,
    'correct': y_test.values == y_test_pred
})
predictions_df.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-predictions.csv', index=False)
print(f"Predictions saved")

print("\n" + "="*70)
print("RANDOM FOREST TRAINING COMPLETE!")
print("="*70)
print(f"All files saved to: {model_dir}")
print(f"Test F1 Score: {test_metrics['f1_score']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")


PART 6: SAVING MODEL AND REPORTS
Model saved: /content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 03-randomforest_model.pkl
Classification report saved
Results summary saved
Predictions saved

RANDOM FOREST TRAINING COMPLETE!
All files saved to: /content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Notebooks/Model_03_Random_Forest_Classifier.ipynb
Test F1 Score: 0.8325
Test Accuracy: 0.8800
